# Notebook 06b — Contextual Embedding Extraction (ClinicalBERT)

---

## Objectives

Extract a single fixed-size (768-d) feature vector per patient report using a frozen, pretrained ClinicalBERT encoder — no fine-tuning anywhere in this notebook.

Pipeline: `clean_report` (already done, Notebook 04) → tokenize (`max_length=256`, padding/truncation) → frozen encoder forward pass → attention-mask-aware mean pooling → L2 normalize.

Train, validation, and test reports are embedded **independently, in separate calls**, using the exact same frozen weights and the exact same `max_length` (chosen from train-only statistics in Notebook 05b — see that notebook's fix note). No information crosses between splits at this stage: there is nothing here that is *fit* on data at all, since the encoder weights are frozen pretrained weights, not parameters estimated from this dataset.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F

from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModel
)

## 1. Load Cleaned Reports


In [2]:
DATASET_DIR = Path(
    "/kaggle/input/datasets/mariammohamed1095/reportssss/datasets/processed/nlp"
)

train_df = pd.read_csv(
    DATASET_DIR /
    "train_reports_clean.csv"
)

val_df = pd.read_csv(
    DATASET_DIR /
    "validation_reports_clean.csv"
)

test_df = pd.read_csv(
    DATASET_DIR /
    "test_reports_clean.csv"
)

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(257, 6)
(56, 6)
(56, 6)


## 2. Model Configuration

`max_length=256`, decided from train-only token-length statistics in Notebook 05b.


In [3]:
MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"

MAX_LENGTH = 256

## 3. Load Frozen Tokenizer & Encoder


In [4]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModel.from_pretrained(
    MODEL_NAME
)

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 4. Device


In [5]:
device = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else "cpu"

)

model.to(device)

model.eval()

print(device)

cuda


## 5. Attention-Mask-Aware Mean Pooling

Averages `last_hidden_state` only over real (non-padding) token positions — naive mean pooling over all 256 positions would be biased by [PAD]-token embeddings, which are not zero vectors (confirmed in Notebook 01 §6). This resolves the pooling decision deferred from earlier notebooks.


In [6]:
def mean_pooling(model_output, attention_mask):

    embeddings = model_output.last_hidden_state

    mask = attention_mask.unsqueeze(-1).expand(
        embeddings.size()
    ).float()

    summed = torch.sum(
        embeddings * mask,
        dim=1
    )

    counted = torch.clamp(
        mask.sum(dim=1),
        min=1e-9
    )

    return summed / counted

## 6. Single-Report Embedding Function

Tokenize → forward pass (no_grad) → mean pool → L2 normalize.


In [7]:
def extract_embedding(text):

    encoded = tokenizer(

        text,

        max_length=MAX_LENGTH,

        truncation=True,

        padding="max_length",

        return_tensors="pt"

    )

    encoded = {

        k: v.to(device)

        for k, v in encoded.items()

    }

    with torch.no_grad():

        outputs = model(**encoded)

        pooled = mean_pooling(

            outputs,

            encoded["attention_mask"]

        )

    pooled = pooled.squeeze()

    pooled = F.normalize(

        pooled,

        p=2,

        dim=0

    )

    return pooled.cpu().numpy()

## 7. Batch Extraction Helper


In [8]:
def extract_dataframe(df):

    features = []

    for report in tqdm(df.clean_report):

        features.append(

            extract_embedding(report)

        )

    return np.vstack(features)

## 8. Extract Embeddings — Train / Validation / Test, Independently


In [9]:
train_embeddings = extract_dataframe(train_df)

validation_embeddings = extract_dataframe(val_df)

test_embeddings = extract_dataframe(test_df)

  0%|          | 0/257 [00:00<?, ?it/s]

  0%|          | 0/56 [00:00<?, ?it/s]

  0%|          | 0/56 [00:00<?, ?it/s]

### 8.1 Shape Check


In [10]:
print(train_embeddings.shape)

print(validation_embeddings.shape)

print(test_embeddings.shape)

(257, 768)
(56, 768)
(56, 768)


## 9. Embedding Sanity Checks (Train)


In [11]:
print("Mean :", train_embeddings.mean())

print("Std  :", train_embeddings.std())

print("NaN  :", np.isnan(train_embeddings).sum())

Mean : -0.0006873994
Std  : 0.036077846
NaN  : 0


### 9.1 Embedding Norms (Should Be ≈ 1.0 After L2 Normalization)


In [12]:
norms = np.linalg.norm(

    train_embeddings,

    axis=1

)

print("Mean Norm :", norms.mean())

print("Min Norm :", norms.min())

print("Max Norm :", norms.max())

Mean Norm : 1.0
Min Norm : 0.9999998
Max Norm : 1.0000001


## 10. Save Embeddings


In [13]:
MODEL_FOLDER = MODEL_NAME.split("/")[-1]

OUTPUT_DIR = Path(

    "/kaggle/working/datasets/processed/nlp"

) / MODEL_FOLDER

OUTPUT_DIR.mkdir(

    parents=True,

    exist_ok=True

)

### 10.1 Save Metadata (patient_id, in the Same Row Order as the .npy Files)

⚠️ Downstream consumers (dataset.py, the fusion module) must join on `patient_id` from this file — never assume positional alignment with the CV module's `.pt` file order.


In [14]:
np.save(
    OUTPUT_DIR /
    "train_embeddings.npy",
    train_embeddings
)

np.save(
    OUTPUT_DIR /
    "validation_embeddings.npy",
    validation_embeddings
)

np.save(
    OUTPUT_DIR /
    "test_embeddings.npy",
    test_embeddings
)

print("Embeddings Saved.")

Embeddings Saved.


In [15]:
train_df[["patient_id"]].to_csv(
    OUTPUT_DIR /
    "train_metadata.csv",
    index=False
)

validation_df = val_df[["patient_id"]]
validation_df.to_csv(
    OUTPUT_DIR /
    "validation_metadata.csv",
    index=False
)

test_df[["patient_id"]].to_csv(
    OUTPUT_DIR /
    "test_metadata.csv",
    index=False
)

print("Metadata Saved.")

Metadata Saved.
